# Phase 2 — Train CodeBERT teacher (save everything to Drive)

Trains a multitask CodeBERT teacher on the competition train split.
Checkpoints + metrics go to:
`MyDrive/NLBSE27_DistilVulSATD/checkpoints/teacher_runs/`

First run suggestion: `INPUT_MODE='raw'`, `MAX_LENGTH=256`, `LOSS='focal'`, `EPOCHS=2`.

In [1]:
!pip -q install "transformers>=4.36.0" "datasets>=2.16.0" "pandas" "numpy" "tqdm" "scikit-learn" "accelerate" "safetensors"

from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/NLBSE27_DistilVulSATD')
(DRIVE_ROOT / 'results').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'logs').mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)

Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/NLBSE27_DistilVulSATD


In [2]:
import json
import re
import time
from dataclasses import asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)
assert device.type == 'cuda', 'Enable T4 GPU first'

device = cuda


In [3]:
# ===== knobs you can change between runs =====
MODEL_NAME = 'microsoft/codebert-base'
INPUT_MODE = 'raw'          # 'raw' or 'bimodal'
MAX_LENGTH = 256
LOSS = 'focal'              # 'focal' or 'bce'
EPOCHS = 2
BATCH_SIZE = 16
LR = 2e-5
VAL_FRACTION = 0.05
SEED = 42
EVAL_PUBLIC_TEST = False    # set True only for a chosen run
QUICK = False               # True = 2000-row smoke train

RUN_NAME = f"codebert_{INPUT_MODE}_L{MAX_LENGTH}_{LOSS}_e{EPOCHS}"
OUT_DIR = DRIVE_ROOT / 'checkpoints' / 'teacher_runs' / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

OUT_DIR = /content/drive/MyDrive/NLBSE27_DistilVulSATD/checkpoints/teacher_runs/codebert_raw_L256_focal_e2


In [4]:
# helpers: labels, bimodal packing, loss, metrics
_BLOCK = re.compile(r'/\*.*?\*/', re.DOTALL)
_LINE = re.compile(r'//.*?$', re.MULTILINE)

def parse_label(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return [int(v[0]), int(v[1])]
    nums = re.findall(r'-?\d+', str(v))
    return [int(nums[0]), int(nums[1])]

def extract_comments_and_code(text):
    comments = []
    def keep(m):
        comments.append(m.group(0).strip()); return ' '
    t = _BLOCK.sub(keep, text or '')
    t = _LINE.sub(keep, t)
    code = re.sub(r'\n{3,}', '\n\n', t).strip()
    return '\n'.join(comments).strip(), code

def format_input(text, mode='raw'):
    if mode == 'raw':
        return text or ''
    comments, code = extract_comments_and_code(text)
    if comments and code:
        return f"{comments}</s>{code}"
    return comments or code or (text or '')

class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_pos=0.0, gamma_neg=3.0, class_weights=None):
        super().__init__()
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.register_buffer('class_weights', None if class_weights is None else class_weights.float())
    def forward(self, logits, targets):
        targets = targets.float()
        probs = torch.sigmoid(logits)
        loss_pos = -((1 - probs) ** self.gamma_pos) * torch.log(probs.clamp_min(1e-8))
        loss_neg = -(probs ** self.gamma_neg) * torch.log((1 - probs).clamp_min(1e-8))
        loss = targets * loss_pos + (1 - targets) * loss_neg
        if self.class_weights is not None:
            loss = loss * self.class_weights.view(1, -1)
        return loss.mean()

def f1_metrics(preds, labels):
    rows = []
    for i, name in enumerate(['Vulnerability', 'MAT']):
        tp = int(((preds[:, i] == 1) & (labels[:, i] == 1)).sum())
        fp = int(((preds[:, i] == 1) & (labels[:, i] == 0)).sum())
        fn = int(((preds[:, i] == 0) & (labels[:, i] == 1)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        rows.append({'cat': name, 'precision': p, 'recall': r, 'f1': f1})
    df = pd.DataFrame(rows)
    return df, float(df.f1.mean())

class FnDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.tok = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, padding='max_length',
                       max_length=self.max_length, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[i])
        return item

print('helpers ready')

helpers ready


In [5]:
# load data + carve val from train only
torch.manual_seed(SEED)
np.random.seed(SEED)

ds = load_dataset('NLBSE/nlbse27-code-comment-classification')
train_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()
train_df['label_list'] = train_df['Label'].apply(parse_label)
test_df['label_list'] = test_df['Label'].apply(parse_label)
train_df['vuln'] = train_df['label_list'].apply(lambda x: x[0])
train_df['satd'] = train_df['label_list'].apply(lambda x: x[1])

if QUICK:
    train_df = train_df.sample(n=2000, random_state=SEED).reset_index(drop=True)
    EPOCHS = 1
    EVAL_PUBLIC_TEST = False

joint = train_df['vuln'].astype(str) + '_' + train_df['satd'].astype(str)
fit_df, val_df = train_test_split(train_df, test_size=VAL_FRACTION, random_state=SEED, stratify=joint)
fit_df = fit_df.reset_index(drop=True); val_df = val_df.reset_index(drop=True)

fit_texts = [format_input(t, INPUT_MODE) for t in fit_df['Function']]
val_texts = [format_input(t, INPUT_MODE) for t in val_df['Function']]
fit_labels = np.stack(fit_df['label_list'].to_numpy())
val_labels = np.stack(val_df['label_list'].to_numpy())

print(len(fit_df), 'train-fit |', len(val_df), 'val | satd+ rate', float(fit_df['satd'].mean()))

README.md:   0%|          | 0.00/530 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 43.3MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.15MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/214586 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/30655 [00:00<?, ? examples/s]

203856 train-fit | 10730 val | satd+ rate 0.01649203359233969


In [6]:
# build model / optim / loaders
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, problem_type='multi_label_classification',
    id2label={0: 'weakness', 1: 'MAT'}, label2id={'weakness': 0, 'MAT': 1},
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
model.to(device)

vuln_rate = float(fit_df['vuln'].mean()); satd_rate = float(fit_df['satd'].mean())
w = torch.tensor([(1/max(vuln_rate,1e-6))**0.5, 1.5*((1/max(satd_rate,1e-6))**0.5)], device=device)
w = w / w.mean()

loss_fn = AsymmetricFocalLoss(0.0, 3.0, w) if LOSS == 'focal' else nn.BCEWithLogitsLoss(pos_weight=w)
fit_loader = DataLoader(FnDataset(fit_texts, fit_labels, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(FnDataset(val_texts, val_labels, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
steps = max(len(fit_loader) * EPOCHS, 1)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06*steps), steps)
scaler = torch.amp.GradScaler('cuda')
print('weights', w.tolist(), 'steps', steps)

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

... (2 outputs left out)

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

weights [0.26011449098587036, 1.7398854494094849] steps 25482


In [7]:
@torch.no_grad()
def predict_texts(texts, labels, bs=64):
    model.eval()
    preds = []
    loader = DataLoader(FnDataset(texts, labels, tokenizer, MAX_LENGTH), batch_size=bs)
    for batch in loader:
        labels_b = batch.pop('labels')
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        preds.append((torch.sigmoid(logits) > 0.5).int().cpu().numpy())
    return np.concatenate(preds, 0), np.asarray(labels)

history = []
best_f1 = -1.0
best_dir = OUT_DIR / 'best_model'

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for batch in tqdm(fit_loader, desc=f'train e{epoch}'):
        y = batch.pop('labels').to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            loss = loss_fn(model(**batch).logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tr_loss += float(loss.item())
    tr_loss /= max(len(fit_loader), 1)

    pred, lab = predict_texts(val_texts, val_labels)
    scores_df, avg_f1 = f1_metrics(pred, lab)
    print(f'epoch {epoch}: train_loss={tr_loss:.4f} val_avgF1={avg_f1:.4f}')
    print(scores_df.to_string(index=False))
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'val_avg_f1': avg_f1,
                    'per_class': scores_df.to_dict(orient='records')})

    if avg_f1 > best_f1:
        best_f1 = avg_f1
        best_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(best_dir)
        tokenizer.save_pretrained(best_dir)
        print('saved best ->', best_dir)

payload = {
    'run_name': RUN_NAME,
    'model_name': MODEL_NAME,
    'input_mode': INPUT_MODE,
    'max_length': MAX_LENGTH,
    'loss': LOSS,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'best_val_f1': best_f1,
    'history': history,
}
with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump(payload, f, indent=2)
with open(DRIVE_ROOT / 'results' / f'teacher_{RUN_NAME}.json', 'w') as f:
    json.dump(payload, f, indent=2)
print('DONE. best_val_f1 =', best_f1)
print('checkpoint:', best_dir)
print('metrics:', OUT_DIR / 'metrics.json')

Streaming output truncated to the last 5000 lines.
train e1:  80%|████████  | 10242/12741 [40:38<09:46,  4.26it/s]

train e1:  80%|████████  | 10243/12741 [40:38<09:52,  4.22it/s]

train e1:  80%|████████  | 10244/12741 [40:38<09:57,  4.18it/s]

train e1:  80%|████████  | 10245/12741 [40:39<09:51,  4.22it/s]

train e1:  80%|████████  | 10246/12741 [40:39<09:53,  4.20it/s]

train e1:  80%|████████  | 10247/12741 [40:39<09:50,  4.23it/s]

train e1:  80%|████████  | 10248/12741 [40:39<09:46,  4.25it/s]

train e1:  80%|████████  | 10249/12741 [40:40<09:46,  4.25it/s]

train e1:  80%|████████  | 10250/12741 [40:40<09:42,  4.27it/s]

train e1:  80%|████████  | 10251/12741 [40:40<09:49,  4.23it/s]

train e1:  80%|████████  | 10252/12741 [40:40<09:44,  4.26it/s]

train e1:  80%|████████  | 10253/12741 [40:41<09:42,  4.27it/s]

train e1:  80%|████████  | 10254/12741 [40:41<09:43,  4.26it/s]

train e1:  80%|████████  | 10255/12741 [40:41<09:47,  4.23it/s]

train e1:  80%|████████  | 10256/12741 

epoch 1: train_loss=0.0295 val_avgF1=0.7894
          cat  precision   recall       f1
Vulnerability   0.832837 0.875213 0.853500
          MAT   0.962617 0.581921 0.725352


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved best -> /content/drive/MyDrive/NLBSE27_DistilVulSATD/checkpoints/teacher_runs/codebert_raw_L256_focal_e2/best_model


train e2: 100%|██████████| 12741/12741 [48:39<00:00,  4.36it/s]


epoch 2: train_loss=0.0217 val_avgF1=0.7922
          cat  precision   recall       f1
Vulnerability   0.816010 0.912735 0.861666
          MAT   0.953704 0.581921 0.722807


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved best -> /content/drive/MyDrive/NLBSE27_DistilVulSATD/checkpoints/teacher_runs/codebert_raw_L256_focal_e2/best_model
DONE. best_val_f1 = 0.7922367302934648
checkpoint: /content/drive/MyDrive/NLBSE27_DistilVulSATD/checkpoints/teacher_runs/codebert_raw_L256_focal_e2/best_model
metrics: /content/drive/MyDrive/NLBSE27_DistilVulSATD/checkpoints/teacher_runs/codebert_raw_L256_focal_e2/metrics.json
